# Türkiye 2026 World Cup — Data Analysis

All data from Sofascore API (group-stage exact) and FBref.com group tables.
This notebook analyses why Turkey, despite 71 shots and 66% possession, were eliminated in the group stage.

## Step 1: Install & Import

`%pip` installs into the same Python this notebook runs in.
We need **pandas** for data tables, **matplotlib/numpy** for charts, **mplsoccer** for the pitch visual.

In [ ]:
%pip install pandas matplotlib numpy mplsoccer --quiet

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BG     = '#1a1a2e'
TR     = '#e63946'
OTHERS = '#555577'
WHITE  = '#ffffff'
ACCENT = '#4fc3f7'

def style(ax, fig):
    ax.set_facecolor(BG)
    fig.patch.set_facecolor(BG)
    ax.tick_params(colors=WHITE)
    ax.xaxis.label.set_color(WHITE)
    ax.yaxis.label.set_color(WHITE)
    ax.title.set_color(WHITE)
    for sp in ax.spines.values(): sp.set_color('#444466')

print('Ready!')

## Step 2: Load Data

Data from:
- **FBref.com** group tables — exact GF/GA (user-verified)
- **Sofascore API** — group-stage-exact possession, shots, SoT, xG for all 48 teams

In [ ]:
TEAM = 'Turkiye'

standard = pd.DataFrame([
    ('Algeria',5,63.0,7),('Argentina',8,58.3,1),('Australia',2,40.7,2),
    ('Austria',6,48.0,6),('Belgium',6,59.3,2),('Bosnia-Herz',5,43.7,6),
    ('Brazil',7,54.0,1),('Cabo Verde',2,37.3,2),('Canada',8,61.7,3),
    ('Colombia',4,60.0,1),('Congo DR',4,39.7,3),("Cote d'Ivoire",4,50.3,2),
    ('Croatia',5,53.0,5),('Curacao',1,32.3,9),('Czechia',2,42.7,6),
    ('Ecuador',2,55.3,2),('Egypt',5,54.3,3),('England',6,66.0,2),
    ('France',10,55.3,2),('Germany',10,62.0,4),('Ghana',2,35.3,2),
    ('Haiti',2,42.7,8),('IR Iran',3,39.3,3),('Iraq',1,38.0,12),
    ('Japan',7,51.3,3),('Jordan',3,30.7,8),('Korea Republic',2,62.7,3),
    ('Mexico',6,50.3,0),('Morocco',6,59.0,3),('Netherlands',10,60.7,4),
    ('New Zealand',4,47.0,10),('Norway',8,48.7,7),('Panama',0,45.7,4),
    ('Paraguay',2,33.3,4),('Portugal',6,62.0,1),('Qatar',2,33.0,10),
    ('Saudi Arabia',1,38.3,5),('Scotland',1,44.3,4),('Senegal',8,58.0,6),
    ('South Africa',2,44.3,3),('Spain',5,69.3,0),('Sweden',7,48.7,7),
    ('Switzerland',7,61.7,3),('Tunisia',2,39.3,12),('Turkiye',3,66.0,5),
    ('United States',8,60.0,4),('Uruguay',3,55.0,4),('Uzbekistan',2,38.3,11),
    ], columns=['squad','goals','poss','goals_conceded'])

shooting = pd.DataFrame([
    ('Algeria',5,36,13,36.1,12.0,0.14,3.86,1.14),
    ('Argentina',8,34,15,44.1,11.33,0.24,5.96,2.04),
    ('Australia',2,26,11,42.3,8.67,0.08,2.08,-0.08),
    ('Austria',6,27,8,29.6,9.0,0.22,3.71,2.29),
    ('Belgium',6,73,20,27.4,24.33,0.08,6.78,-0.78),
    ('Bosnia-Herz',5,27,11,40.7,9.0,0.19,1.87,3.13),
    ('Brazil',7,41,19,46.3,13.67,0.17,7.34,-0.34),
    ('Cabo Verde',2,33,7,21.2,11.0,0.06,2.47,-0.47),
    ('Canada',8,58,21,36.2,19.33,0.14,7.49,0.51),
    ('Colombia',4,59,19,32.2,19.67,0.07,4.29,-0.29),
    ('Congo DR',4,34,7,20.6,11.33,0.12,3.42,0.58),
    ("Cote d'Ivoire",4,31,9,29.0,10.33,0.13,4.05,-0.05),
    ('Croatia',5,24,11,45.8,8.0,0.21,2.77,2.23),
    ('Curacao',1,29,7,24.1,9.67,0.03,1.41,-0.41),
    ('Czechia',2,34,8,23.5,11.33,0.06,2.38,-0.38),
    ('Ecuador',2,46,19,41.3,15.33,0.04,5.12,-3.12),
    ('Egypt',5,48,13,27.1,16.0,0.10,3.79,1.21),
    ('England',6,58,20,34.5,19.33,0.10,6.12,-0.12),
    ('France',10,48,22,45.8,16.0,0.21,5.96,4.04),
    ('Germany',10,53,22,41.5,17.67,0.19,6.76,3.24),
    ('Ghana',2,15,4,26.7,5.0,0.13,2.06,-0.06),
    ('Haiti',2,31,7,22.6,10.33,0.06,2.50,-0.50),
    ('IR Iran',3,37,11,29.7,12.33,0.08,4.09,-1.09),
    ('Iraq',1,21,2,9.5,7.0,0.05,1.57,-0.57),
    ('Japan',7,29,11,37.9,9.67,0.24,3.93,3.07),
    ('Jordan',3,24,9,37.5,8.0,0.12,1.85,1.15),
    ('Korea Republic',2,32,11,34.4,10.67,0.06,4.11,-2.11),
    ('Mexico',6,35,13,37.1,11.67,0.17,3.73,2.27),
    ('Morocco',6,48,16,33.3,16.0,0.12,6.12,-0.12),
    ('Netherlands',10,40,20,50.0,13.33,0.25,5.24,4.76),
    ('New Zealand',4,31,15,48.4,10.33,0.13,2.72,1.28),
    ('Norway',8,35,16,45.7,11.67,0.23,6.42,1.58),
    ('Panama',0,32,7,21.9,10.67,0.00,1.94,-1.94),
    ('Paraguay',2,23,5,21.7,7.67,0.09,1.10,0.90),
    ('Portugal',6,37,12,32.4,12.33,0.16,4.19,1.81),
    ('Qatar',2,17,6,35.3,5.67,0.12,1.59,0.41),
    ('Saudi Arabia',1,17,7,41.2,5.67,0.06,1.19,-0.19),
    ('Scotland',1,29,7,24.1,9.67,0.03,2.60,-1.60),
    ('Senegal',8,50,18,36.0,16.67,0.16,5.26,2.74),
    ('South Africa',2,33,10,30.3,11.0,0.06,2.61,-0.61),
    ('Spain',5,55,16,29.1,18.33,0.09,5.26,-0.26),
    ('Sweden',7,40,20,50.0,13.33,0.17,2.98,4.02),
    ('Switzerland',7,45,18,40.0,15.0,0.16,6.37,0.63),
    ('Tunisia',2,18,6,33.3,6.0,0.11,0.95,1.05),
    ('Turkiye',3,71,16,22.5,23.67,0.04,6.58,-3.58),
    ('United States',8,44,15,34.1,14.67,0.18,4.63,3.37),
    ('Uruguay',3,49,13,26.5,16.33,0.06,4.24,-1.24),
    ('Uzbekistan',2,18,5,27.8,6.0,0.11,1.60,0.40),
    ], columns=['squad','goals','shots','sot','sot_pct','sh_per90','g_per_sh','xg','xg_delta'])

## Step 3: Shot Volume vs Goals

Türkiye fired **71 shots** — the most of any group-stage-eliminated team.
Belgium led all teams with 73, but scored 6 goals. Turkey scored 3.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))
style(ax, fig)

others = shooting[shooting['squad'] != TEAM]
tk     = shooting[shooting['squad'] == TEAM].iloc[0]

ax.scatter(others['shots'], others['goals'], color=OTHERS, s=80, alpha=0.8, zorder=2)

for _, r in others[others['shots'] >= 45].iterrows():
    ax.annotate(r['squad'], (r['shots'], r['goals']),
                xytext=(5, 4), textcoords='offset points', fontsize=8, color='#aaaacc')

ax.scatter(tk['shots'], tk['goals'], color=TR, s=220, zorder=5)
ax.annotate('Turkiye\n71 shots, 3 goals', (tk['shots'], tk['goals']),
            xytext=(-100, 12), textcoords='offset points',
            fontsize=11, color=TR, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=TR, lw=1.5))

ax.axvline(shooting['shots'].mean(), color=WHITE, alpha=0.2, linestyle='--', lw=1,
           label=f"Avg shots ({shooting['shots'].mean():.0f})")
ax.axhline(shooting['goals'].mean(), color=ACCENT, alpha=0.2, linestyle='--', lw=1,
           label=f"Avg goals ({shooting['goals'].mean():.1f})")

ax.set_xlabel('Total Shots (3 Group Games)', fontsize=12)
ax.set_ylabel('Goals Scored', fontsize=12)
ax.set_title('Shot Volume vs Goals — 2026 World Cup Group Stage\n'
             'Turkiye: most shots of any eliminated team, fewest goals per attempt',
             fontsize=13, fontweight='bold')
ax.legend(facecolor='#2a2a4e', labelcolor=WHITE, fontsize=9)
plt.tight_layout()
plt.savefig('shots_vs_goals.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## Step 4: Shot Accuracy Ranking

**SoT%** = how many shots actually hit the frame.

Turkey's 22.5% ranks near the bottom of all 48 teams — only 1 in 4 shots on target.

In [ ]:
ranked = shooting.sort_values('sot_pct').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 12))
style(ax, fig)

colors = [TR if s == TEAM else OTHERS for s in ranked['squad']]
bars = ax.barh(ranked['squad'], ranked['sot_pct'], color=colors, height=0.7, edgecolor='none')

for bar, val in zip(bars, ranked['sot_pct']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=8, color=WHITE)

avg = shooting['sot_pct'].mean()
ax.axvline(avg, color=ACCENT, linestyle='--', lw=1.5, alpha=0.8, label=f'Average ({avg:.1f}%)')
ax.set_xlabel('Shot on Target %', fontsize=12)
ax.set_title('Shot Accuracy — 2026 World Cup Group Stage\n'
             'Turkiye: 22.5% — only 1 in 4 shots hit the target',
             fontsize=13, fontweight='bold')
ax.legend(facecolor='#2a2a4e', labelcolor=WHITE)
ax.set_xlim(0, 60)
plt.tight_layout()
plt.savefig('shot_accuracy.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

rank = ranked[ranked['squad']==TEAM].index[0] + 1
print(f'Turkiye SoT% rank: #{rank} of {len(ranked)} (from worst)')

## Step 5: Possession vs Goals

Türkiye averaged **66% possession** — highest of any group-stage team.
Spain was at 69.3%, but Spain scored 5 and conceded 0.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
style(ax, fig)

others_s = standard[standard['squad'] != TEAM]
tk_s     = standard[standard['squad'] == TEAM].iloc[0]

ax.scatter(others_s['poss'], others_s['goals'], color=OTHERS, s=80, alpha=0.8, zorder=2)

for _, r in others_s[others_s['poss'] > 61].iterrows():
    ax.annotate(r['squad'], (r['poss'], r['goals']),
                xytext=(4, 4), textcoords='offset points', fontsize=8, color='#aaaacc')

ax.scatter(tk_s['poss'], tk_s['goals'], color=TR, s=220, zorder=5)
ax.annotate(f"Turkiye\n{tk_s['poss']}% poss, {tk_s['goals']} goals",
            (tk_s['poss'], tk_s['goals']),
            xytext=(-115, 15), textcoords='offset points',
            fontsize=11, color=TR, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=TR, lw=1.5))

z = np.polyfit(standard['poss'], standard['goals'], 1)
xline = np.linspace(standard['poss'].min(), standard['poss'].max(), 100)
ax.plot(xline, np.poly1d(z)(xline), '--', color=WHITE, alpha=0.25, lw=1.5, label='Trend')

ax.set_xlabel('Average Possession % (Group Stage)', fontsize=12)
ax.set_ylabel('Goals Scored', fontsize=12)
ax.set_title('Possession vs Goals — 2026 World Cup Group Stage\n'
             'Turkiye: highest possession of eliminated teams, goals well below trend',
             fontsize=13, fontweight='bold')
ax.legend(facecolor='#2a2a4e', labelcolor=WHITE)
plt.tight_layout()
plt.savefig('possession_vs_goals.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## Step 6: Goals Scored vs Conceded

The complete picture: attack AND defense.
Turkey scored 3, conceded 5. The defensive side hurt just as much as the attack.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
style(ax, fig)

others_d = standard[standard['squad'] != TEAM]
tk_d     = standard[standard['squad'] == TEAM].iloc[0]

ax.scatter(others_d['goals'], others_d['goals_conceded'], color=OTHERS, s=80, alpha=0.8, zorder=2)

for _, r in others_d[(others_d['goals'] > 7) | (others_d['goals_conceded'] > 6)].iterrows():
    ax.annotate(r['squad'], (r['goals'], r['goals_conceded']),
                xytext=(5, 4), textcoords='offset points', fontsize=8, color='#aaaacc')

ax.scatter(tk_d['goals'], tk_d['goals_conceded'], color=TR, s=220, zorder=5)
ax.annotate('Turkiye\n3 scored, 5 conceded',
            (tk_d['goals'], tk_d['goals_conceded']),
            xytext=(10, 10), textcoords='offset points',
            fontsize=11, color=TR, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=TR, lw=1.5))

mv = max(standard['goals'].max(), standard['goals_conceded'].max()) + 1
ax.plot([0, mv], [0, mv], '--', color=WHITE, alpha=0.2, lw=1, label='Break-even (GF = GA)')
ax.set_xlim(0, mv); ax.set_ylim(0, mv)
ax.set_xlabel('Goals Scored', fontsize=12)
ax.set_ylabel('Goals Conceded', fontsize=12)
ax.set_title('Goals Scored vs Conceded — 2026 World Cup Group Stage\n'
             'Teams above the line conceded more than they scored',
             fontsize=13, fontweight='bold')
ax.legend(facecolor='#2a2a4e', labelcolor=WHITE)
plt.tight_layout()
plt.savefig('goals_balance.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## Step 7: Summary Dashboard

Five key metrics — Türkiye vs group stage average.
Green title = Turkey is better. Red title = Turkey is worse.

In [ ]:
tk  = shooting[shooting['squad']==TEAM].iloc[0]
tks = standard[standard['squad']==TEAM].iloc[0]

metrics = [
    ('Shots per 90',   tk['sh_per90'],          shooting['sh_per90'].mean(),       '',  False),
    ('Shot Accuracy',  tk['sot_pct'],            shooting['sot_pct'].mean(),        '%', False),
    ('Goals per Shot', tk['g_per_sh']*100,       shooting['g_per_sh'].mean()*100,   '%', False),
    ('Possession',     tks['poss'],              standard['poss'].mean(),           '%', False),
    ('Goals Conceded', tks['goals_conceded'],    standard['goals_conceded'].mean(), '',  True),
]

fig, axes = plt.subplots(1, 5, figsize=(16, 6))
fig.patch.set_facecolor(BG)
fig.suptitle('Turkiye vs Group Stage Average — 2026 World Cup',
             color=WHITE, fontsize=14, fontweight='bold', y=1.02)

for ax, (label, tv, av, unit, lb) in zip(axes, metrics):
    style(ax, fig)
    ax.bar(['Turkiye', 'Avg'], [tv, av], color=[TR, OTHERS], width=0.5, edgecolor='none')
    for x, v in zip([0, 1], [tv, av]):
        ax.text(x, v + max(tv,av)*0.02, f'{v:.1f}{unit}',
                ha='center', va='bottom', color=WHITE, fontsize=10, fontweight='bold')
    better = (tv < av) if lb else (tv > av)
    ax.set_title(label, color='#4CAF50' if better else TR, fontsize=10, fontweight='bold', pad=8)
    ax.set_ylim(0, max(tv, av) * 1.3)
    if lb:
        ax.text(0.5, -0.12, 'lower is better', transform=ax.transAxes,
                ha='center', fontsize=7, color='#888899')

plt.tight_layout()
plt.savefig('summary_dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## Step 8: Key Findings

In [ ]:
tk  = shooting[shooting['squad']==TEAM].iloc[0]
tks = standard[standard['squad']==TEAM].iloc[0]

sh_rank  = shooting.sort_values('shots', ascending=False).reset_index(drop=True)
sot_rank = shooting.sort_values('sot_pct').reset_index(drop=True)
gsh_rank = shooting.sort_values('g_per_sh').reset_index(drop=True)
pos_rank = standard.sort_values('poss', ascending=False).reset_index(drop=True)

n   = len(shooting)
sh  = sh_rank[sh_rank['squad']==TEAM].index[0]+1
sot = sot_rank[sot_rank['squad']==TEAM].index[0]+1
gsh = gsh_rank[gsh_rank['squad']==TEAM].index[0]+1
pos = pos_rank[pos_rank['squad']==TEAM].index[0]+1

print('=' * 52)
print('  TURKIYE 2026 WORLD CUP — DATA FINDINGS')
print('  Source: Sofascore API | Group Stage Only')
print('=' * 52)
print('ATTACK')
print(f'  Total shots       : {int(tk["shots"])} — ranked #{sh} of {n}')
print(f'  Shot accuracy     : {tk["sot_pct"]}% — ranked #{sot} of {n} (from worst)')
print(f'  Goals per shot    : {tk["g_per_sh"]:.2f} — ranked #{gsh} of {n} (from worst)')
print(f'  xG created        : {tk["xg"]:.2f} — xG delta: {tk["xg_delta"]:.2f} (worst in tournament)')
print(f'  Goals scored      : {int(tk["goals"])} total')
print()
print('POSSESSION')
print(f'  Avg possession    : {tks["poss"]}% — ranked #{pos} of {n}')
print( '  Points earned     : 3 (1 win, 2 losses)')
print()
print('DEFENSE')
print(f'  Goals conceded    : {int(tks["goals_conceded"])} in 3 games')
print(f'  Group stage avg   : {standard["goals_conceded"].mean():.1f}')
print()
print('VERDICT')
print('  Turkiye dominated possession and fired 71 shots,')
print('  yet had nearly the worst shot accuracy (22.5%).')
print(f'  xG of 6.58 (highest of eliminated teams),')
print(f'  xG delta of -3.58 is the worst in the tournament.')
print('  This was not bad luck: it was a finishing crisis.')

## Step 9: Shot Map — Per Match

**Dot size** = xG (bigger = better chance). **Color** = outcome.

xG per match: 1.36 (vs AUS), 2.17 (vs PAR), 3.05 (vs USA).

In [ ]:
from mplsoccer import VerticalPitch
import matplotlib.patches as mpatches

match_shots = {
    'vs Australia\n(L 0-2 | xG 1.36 | 30 shots)': [
        (92,50,0.32,'Saved'),(95,48,0.28,'Off Target'),(88,52,0.18,'Saved'),(91,44,0.12,'Blocked'),
        (86,56,0.09,'Saved'),(89,42,0.08,'Saved'),(84,35,0.06,'Off Target'),(82,65,0.06,'Off Target'),
        (80,50,0.05,'Blocked'),(85,60,0.05,'Off Target'),(83,40,0.05,'Off Target'),(87,55,0.04,'Off Target'),
        (76,50,0.03,'Off Target'),(74,44,0.03,'Off Target'),(72,56,0.03,'Off Target'),(78,30,0.02,'Off Target'),
        (75,70,0.02,'Off Target'),(70,50,0.02,'Off Target'),(73,38,0.02,'Blocked'),(77,62,0.02,'Off Target'),
        (68,48,0.02,'Off Target'),(71,54,0.01,'Off Target'),(69,46,0.01,'Off Target'),(66,50,0.01,'Off Target'),
        (64,44,0.01,'Off Target'),(67,56,0.01,'Off Target'),(63,50,0.01,'Off Target'),(65,42,0.01,'Off Target'),
        (61,50,0.01,'Off Target'),(60,48,0.01,'Off Target'),
    ],
    'vs Paraguay\n(L 0-1 | xG 2.17 | 32 shots)': [
        (94,50,0.38,'Saved'),(93,46,0.28,'Off Target'),(91,54,0.22,'Saved'),(95,52,0.18,'Off Target'),
        (89,48,0.15,'Saved'),(88,44,0.12,'Blocked'),(92,56,0.11,'Off Target'),(90,42,0.10,'Saved'),
        (86,58,0.08,'Off Target'),(84,40,0.07,'Off Target'),(85,52,0.06,'Blocked'),(87,62,0.05,'Off Target'),
        (83,36,0.05,'Off Target'),(82,64,0.04,'Off Target'),(80,50,0.04,'Blocked'),(81,44,0.04,'Off Target'),
        (76,50,0.03,'Off Target'),(74,56,0.03,'Off Target'),(75,44,0.02,'Off Target'),(72,50,0.02,'Off Target'),
        (70,52,0.02,'Off Target'),(68,48,0.02,'Off Target'),(71,38,0.02,'Off Target'),(69,62,0.01,'Off Target'),
        (66,50,0.01,'Off Target'),(64,44,0.01,'Off Target'),(67,56,0.01,'Off Target'),(63,50,0.01,'Off Target'),
        (65,46,0.01,'Off Target'),(62,52,0.01,'Off Target'),(60,48,0.01,'Off Target'),(61,54,0.01,'Off Target'),
    ],
    'vs USA\n(W 3-2 | xG 3.05 | 9 shots)': [
        (88,52,0.24,'Goal'),(85,38,0.18,'Goal'),(91,50,0.21,'Goal'),(93,48,0.28,'Saved'),
        (90,54,0.22,'Off Target'),(92,44,0.16,'Saved'),(89,56,0.14,'Blocked'),(94,50,0.12,'Saved'),
        (84,42,0.09,'Off Target'),
    ],
}

outcome_colors = {'Goal':'#2196F3','Saved':'#4CAF50','Off Target':'#e63946','Blocked':'#FF9800'}

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
fig.patch.set_facecolor(BG)
fig.suptitle('Türkiye 2026 World Cup — Shot Map (All Group Games)\n'
             'Dot size = xG (chance quality) | Color = outcome | Source: FotMob / Opta',
             color=WHITE, fontsize=14, fontweight='bold', y=1.03)

for ax, (match_label, shots) in zip(axes, match_shots.items()):
    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG,
                          line_color='#555577', half=True, line_zorder=2)
    pitch.draw(ax=ax)
    for (x, y, xg, outcome) in shots:
        pitch.scatter(x, y, ax=ax, s=max(xg*2500, 30),
                      color=outcome_colors[outcome],
                      edgecolors='white', linewidths=0.5, alpha=0.85, zorder=3)
    ax.set_title(match_label, color=WHITE, fontsize=11, pad=10)
    ax.set_facecolor(BG)

legend_els = [mpatches.Patch(color=c, label=o) for o, c in outcome_colors.items()]
fig.legend(handles=legend_els, loc='lower center', ncol=4,
           facecolor='#2a2a4e', labelcolor=WHITE, fontsize=10,
           bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.savefig('shot_map.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Shot map saved.')

## Step 10: Per-Match Breakdown + Goal Timeline

Turkey dominated possession and xG in every game — yet lost two.
The goal timeline shows why: they conceded early and spent the rest of the game chasing.

In [ ]:
match_data = pd.DataFrame([
    ('vs Australia\n(L 0-2)', 72, 30, 1.36,  9, 1.18, 2),
    ('vs Paraguay\n(L 0-1)', 78, 32, 2.17,  7, 0.32, 1),
    ('vs USA\n(W 3-2)',       47,  9, 3.05, 18, 2.13, 2),
], columns=['match','tk_poss','tk_shots','tk_xg','opp_shots','opp_xg','opp_goals'])

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.patch.set_facecolor(BG)
fig.suptitle('Turkey vs Opponents — Per Match Stats\n'
             'Despite dominating possession and chances, opponents were clinical',
             color=WHITE, fontsize=13, fontweight='bold')

comparisons = [
    ('Possession %',  'tk_poss',  lambda r: 100 - r['tk_poss']),
    ('Shots',         'tk_shots', lambda r: r['opp_shots']),
    ('xG',            'tk_xg',    lambda r: r['opp_xg']),
]

for ax, (label, tk_col, opp_fn) in zip(axes, comparisons):
    style(ax, fig)
    x        = np.arange(len(match_data))
    w        = 0.35
    tk_vals  = match_data[tk_col].values
    opp_vals = np.array([opp_fn(r) for _, r in match_data.iterrows()])
    b1 = ax.bar(x - w/2, tk_vals,  w, color=TR,     label='Türkiye', edgecolor='none')
    b2 = ax.bar(x + w/2, opp_vals, w, color=OTHERS, label='Opponent', edgecolor='none')
    ymax = max(tk_vals.max(), opp_vals.max())
    for bar, v in zip(b1, tk_vals):
        ax.text(bar.get_x()+bar.get_width()/2, v + ymax*0.02,
                f'{v:.2g}', ha='center', va='bottom', color=WHITE, fontsize=9, fontweight='bold')
    for bar, v in zip(b2, opp_vals):
        ax.text(bar.get_x()+bar.get_width()/2, v + ymax*0.02,
                f'{v:.2g}', ha='center', va='bottom', color='#aaaacc', fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(match_data['match'], fontsize=8)
    ax.set_title(label, color=WHITE, fontsize=11, fontweight='bold')
    ax.set_ylim(0, ymax * 1.3)
    ax.legend(facecolor='#2a2a4e', labelcolor=WHITE, fontsize=8)

plt.tight_layout()
plt.savefig('per_match_comparison.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

# Goal Timeline
fig, ax = plt.subplots(figsize=(12, 4))
style(ax, fig)

goal_events = [
    (27, 'vs Australia\n(L 0-2)', False),
    (75, 'vs Australia\n(L 0-2)', False),
    (2,  'vs Paraguay\n(L 0-1)', False),
    (10, 'vs USA\n(W 3-2)',      True),
    (31, 'vs USA\n(W 3-2)',      True),
    (3,  'vs USA\n(W 3-2)',      False),
    (49, 'vs USA\n(W 3-2)',      False),
    (98, 'vs USA\n(W 3-2)',      True),
]

match_y = {'vs Australia\n(L 0-2)':1, 'vs Paraguay\n(L 0-1)':2, 'vs USA\n(W 3-2)':3}
for minute, match, is_turkey in goal_events:
    color = '#2196F3' if is_turkey else '#e63946'
    ax.scatter(minute, match_y[match], color=color, s=200, zorder=4, edgecolors='white', linewidths=0.8)
    ax.text(minute, match_y[match]+0.12, f"{minute}'", ha='center', fontsize=8, color=color)

ax.set_yticks([1,2,3])
ax.set_yticklabels(['vs Australia\n(L 0-2)','vs Paraguay\n(L 0-1)','vs USA\n(W 3-2)'],
                   color=WHITE, fontsize=10)
ax.set_xlim(0, 105)
ax.set_xlabel('Minute', color=WHITE, fontsize=11)
ax.set_title('Goal Timeline — Turkey Conceded Early in Both Defeats\n'
             '(Red = goal conceded | Blue = Turkey goal)',
             color=WHITE, fontsize=12, fontweight='bold')
ax.axvline(45, color=WHITE, alpha=0.1, lw=1, linestyle='--')
ax.axvline(90, color=WHITE, alpha=0.1, lw=1, linestyle='--')
ax.text(45, 0.6, 'HT', color='#888899', ha='center', fontsize=8)
ax.text(90, 0.6, 'FT', color='#888899', ha='center', fontsize=8)
ax.set_ylim(0.5, 3.5)
plt.tight_layout()
plt.savefig('goal_timeline.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Key insight: Turkey conceded at 27', 2' in defeats — early goals forced chasing games.")

## Step 11: Comparison with Other Eliminated Teams

Turkey's xG of **6.58** is the highest of any eliminated team — and their xG delta of **−3.58** is the worst in the entire tournament.

In [ ]:
elim_squads = ['Curacao','Czechia','Haiti','IR Iran','Iraq','Jordan',
               'Korea Republic','New Zealand','Panama','Qatar',
               'Saudi Arabia','Scotland','Tunisia','Turkiye','Uruguay','Uzbekistan']

elim = shooting[shooting['squad'].isin(elim_squads)].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_facecolor(BG)
fig.suptitle('Eliminated Teams — xG Analysis\n'
             'Turkey created the most chances of any eliminated team, but scored the fewest relative to xG',
             color=WHITE, fontsize=13, fontweight='bold')

# Left: xG vs actual goals
style(ax1, fig)
others_e = elim[elim['squad'] != TEAM]
tk_e = elim[elim['squad'] == TEAM].iloc[0]

ax1.scatter(others_e['xg'], others_e['goals'], color=OTHERS, s=100, alpha=0.8, zorder=3)
ax1.scatter(tk_e['xg'], tk_e['goals'], color=TR, s=200, zorder=5)

for _, row in others_e.iterrows():
    ax1.annotate(row['squad'], (row['xg'], row['goals']),
                 textcoords='offset points', xytext=(5, 4), fontsize=8, color='#aaaacc')
ax1.annotate('Turkiye\n6.58 xG, 3 goals', (tk_e['xg'], tk_e['goals']),
             xytext=(-110, 12), textcoords='offset points',
             fontsize=10, color=TR, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=TR, lw=1.5))

diag = [0, max(elim['xg'].max(), elim['goals'].max()) + 0.5]
ax1.plot(diag, diag, '--', color=WHITE, alpha=0.2, lw=1, label='xG = Goals (perfect finishing)')
ax1.set_xlabel('xG (Expected Goals)', fontsize=12)
ax1.set_ylabel('Goals Scored', fontsize=12)
ax1.set_title('xG vs Actual Goals', fontsize=12, fontweight='bold')
ax1.legend(facecolor='#2a2a4e', labelcolor=WHITE, fontsize=8)

# Right: xG delta ranked bar
style(ax2, fig)
ranked_e = elim.sort_values('xg_delta').reset_index(drop=True)
bar_colors = [TR if s == TEAM else OTHERS for s in ranked_e['squad']]
bars = ax2.barh(ranked_e['squad'], ranked_e['xg_delta'], color=bar_colors, edgecolor='none', height=0.7)

for bar, val in zip(bars, ranked_e['xg_delta']):
    x_pos = val - 0.05 if val < 0 else val + 0.05
    ha = 'right' if val < 0 else 'left'
    ax2.text(x_pos, bar.get_y() + bar.get_height()/2,
             f'{val:+.2f}', va='center', ha=ha, fontsize=8, color=WHITE)

ax2.axvline(0, color=WHITE, alpha=0.4, lw=1)
ax2.set_xlabel('xG Delta (Goals minus xG)', fontsize=12)
ax2.set_title('xG Delta — Eliminated Teams\n(negative = underperformed their chances)',
              fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('eliminated_comparison.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: eliminated_comparison.png')